In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
from __future__ import annotations

from pathlib import Path
import re
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cuda')

In [18]:
BASE = Path('/content/drive/MyDrive/DS340W')

def resolve_file(*patterns):
    for pattern in patterns:
        matches = sorted(BASE.rglob(pattern))
        if matches:
            print(f'Resolved {pattern}: {matches[0]}')
            return matches[0]
    raise FileNotFoundError(patterns)

DATA_PATH = resolve_file('Updated_Crime_Enriched_PointLevel_v16c.csv', 'final approach/Updated_Crime_Enriched_PointLevel_v16c.csv', '*v16c*.csv')
ARTIFACT_DIR = BASE / 'final approach' / 'parent_paper_v16c_outputs'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PANEL_OUT = ARTIFACT_DIR / 'parent_paper_precinct_panel_v16c.csv'
PRED_OUT = ARTIFACT_DIR / 'parent_paper_predictions_v16c.csv'
SUMMARY_OUT = ARTIFACT_DIR / 'parent_paper_summary_v16c.csv'

LOOKBACK = 8
K_NEIGHBORS = 5
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
PATIENCE = 25

FULL_START = pd.Timestamp('2015-01-01')
FULL_END = pd.Timestamp('2020-03-10')
TRAIN_END = pd.Timestamp('2019-12-31')
TEST_START = pd.Timestamp('2020-01-01')
TEST_END = pd.Timestamp('2020-01-07')

print('DATA_PATH =', DATA_PATH)
print('ARTIFACT_DIR =', ARTIFACT_DIR)

Resolved Updated_Crime_Enriched_PointLevel_v16c.csv: /content/drive/MyDrive/DS340W/Updated_Crime_Enriched_PointLevel_v16c.csv
DATA_PATH = /content/drive/MyDrive/DS340W/Updated_Crime_Enriched_PointLevel_v16c.csv
ARTIFACT_DIR = /content/drive/MyDrive/DS340W/final approach/parent_paper_v16c_outputs


In [19]:
def clean_precinct(x):
    if pd.isna(x):
        return None
    s = re.sub(r'[^0-9]', '', str(x))
    return s if s else None

def map_crime_type(desc):
    d = str(desc).upper()
    if any(k in d for k in ['ASSAULT', 'HARRASSMENT', 'HARASSMENT', 'OFFENSES AGAINST THE PERSON']):
        return 'ASSAULT'
    if 'ROBBERY' in d:
        return 'ROBBERY'
    if any(k in d for k in ['PETIT LARCENY', 'GRAND LARCENY', 'LARCENY', 'THEFT', 'STOLEN PROPERTY']):
        return 'THEFT'
    if any(k in d for k in ['CRIMINAL MISCHIEF', 'CRIMINAL DAMAGE', 'DAMAGE', 'VANDAL']):
        return 'CRIMINAL_DAMAGE'
    return None

def build_knn_adjacency(coords: np.ndarray, k: int = 4) -> np.ndarray:
    n = coords.shape[0]
    adj = np.zeros((n, n), dtype=np.float32)
    dists = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(-1))
    for i in range(n):
        nn_idx = np.argsort(dists[i])[1:k+1]
        adj[i, nn_idx] = 1.0
    adj = np.maximum(adj, adj.T)
    np.fill_diagonal(adj, 1.0)
    return adj

def normalize_adj(adj: np.ndarray) -> np.ndarray:
    d = adj.sum(axis=1)
    d_inv_sqrt = np.power(d, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    D = np.diag(d_inv_sqrt)
    return D @ adj @ D

def rmse_torch(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2))

In [20]:
usecols = [
    'row_id', 'crime_date', 'crime_precinct', 'addr_pct_cd', 'ofns_desc',
    'weekend', 'holiday', 't_mean', 'rh_mean', 'wind_mean', 'weather_obs_time',
    'is_christian_holiday', 'is_islamic_holiday', 'is_jewish_holiday', 'is_hindu_holiday',
    'event_match_count', 'has_event', 'road_closure_count', 'has_road_closure',
    'latitude', 'longitude'
]

raw = pd.read_csv(DATA_PATH, usecols=lambda c: c in usecols, low_memory=False)
if 'row_id' in raw.columns:
    raw = raw.loc[pd.to_numeric(raw['row_id'], errors='coerce').fillna(-1) != 0].copy()

raw['crime_date'] = pd.to_datetime(raw['crime_date'], errors='coerce')
raw = raw[(raw['crime_date'] >= FULL_START) & (raw['crime_date'] <= FULL_END)].copy()
raw['precinct'] = raw['crime_precinct'].map(clean_precinct)
raw['crime_type'] = raw['ofns_desc'].map(map_crime_type)
raw = raw.dropna(subset=['crime_date', 'precinct', 'crime_type'])

raw['month'] = raw['crime_date'].dt.month
raw['dow'] = raw['crime_date'].dt.dayofweek
raw['is_weekend'] = (raw['dow'] >= 5).astype(int)
if 'weekend' not in raw.columns:
    raw['weekend'] = raw['is_weekend']
else:
    raw['weekend'] = raw['weekend'].fillna(raw['is_weekend']).astype(int)
raw['holiday'] = raw['holiday'].fillna(0).astype(int)

raw[['event_match_count', 'has_event', 'road_closure_count', 'has_road_closure']] = raw[['event_match_count', 'has_event', 'road_closure_count', 'has_road_closure']].fillna(0)

raw[['latitude', 'longitude']] = raw[['latitude', 'longitude']].apply(pd.to_numeric, errors='coerce')
raw[['t_mean', 'rh_mean', 'wind_mean']] = raw[['t_mean', 'rh_mean', 'wind_mean']].apply(pd.to_numeric, errors='coerce')

raw[['is_christian_holiday', 'is_islamic_holiday', 'is_jewish_holiday', 'is_hindu_holiday']] = raw[['is_christian_holiday', 'is_islamic_holiday', 'is_jewish_holiday', 'is_hindu_holiday']].fillna(0).astype(int)

print(raw.shape)
raw[['crime_date', 'precinct', 'crime_type', 'ofns_desc']].head()

(1650448, 26)


,crime_date,precinct,crime_type,ofns_desc
3,2015-01-01,1050,THEFT,THEFT-FRAUD
17,2015-01-01,100,THEFT,THEFT-FRAUD
38,2015-01-01,900,THEFT,GRAND LARCENY
58,2015-01-01,750,ASSAULT,HARRASSMENT 2
65,2015-01-01,1050,THEFT,PETIT LARCENY


In [21]:
panel = raw.groupby(['crime_date', 'precinct', 'crime_type'], as_index=False).agg(
    crime_count=('ofns_desc', 'size'),
    weekend=('weekend', 'max'),
    holiday=('holiday', 'max'),
    t_mean=('t_mean', 'mean'),
    rh_mean=('rh_mean', 'mean'),
    wind_mean=('wind_mean', 'mean'),
    has_event=('has_event', 'max'),
    event_match_count=('event_match_count', 'max'),
    has_road_closure=('has_road_closure', 'max'),
    road_closure_count=('road_closure_count', 'max'),
    is_christian_holiday=('is_christian_holiday', 'max'),
    is_islamic_holiday=('is_islamic_holiday', 'max'),
    is_jewish_holiday=('is_jewish_holiday', 'max'),
    is_hindu_holiday=('is_hindu_holiday', 'max'),
    month=('month', 'first'),
    dow=('dow', 'first')
)

dates = pd.date_range(panel['crime_date'].min(), panel['crime_date'].max(), freq='D')
precincts = sorted(panel['precinct'].astype(str).unique())
crime_types = sorted(panel['crime_type'].unique())

grid = pd.MultiIndex.from_product([dates, precincts, crime_types], names=['crime_date', 'precinct', 'crime_type'])
panel = panel.set_index(['crime_date', 'precinct', 'crime_type']).reindex(grid).reset_index()

panel['crime_count'] = panel['crime_count'].fillna(0)
for col in ['weekend', 'holiday', 'has_event', 'event_match_count', 'has_road_closure', 'road_closure_count', 'is_christian_holiday', 'is_islamic_holiday', 'is_jewish_holiday', 'is_hindu_holiday']:
    panel[col] = panel[col].fillna(0)

panel['month'] = panel['crime_date'].dt.month
panel['dow'] = panel['crime_date'].dt.dayofweek

for col in ['t_mean', 'rh_mean', 'wind_mean']:
    panel[col] = panel.groupby('crime_date')[col].transform(lambda s: s.fillna(s.mean()))
    panel[col] = panel[col].fillna(0)

panel = panel.sort_values(['precinct', 'crime_type', 'crime_date']).reset_index(drop=True)
panel['e_mean'] = (panel['rh_mean'] / 100.0) * 6.105 * np.exp((17.2 * panel['t_mean']) / (237.7 + panel['t_mean']))
panel['weekday_avg'] = panel.groupby(['precinct', 'crime_type', 'dow'])['crime_count'].transform('mean')
panel['weekend_avg'] = panel.groupby(['precinct', 'crime_type', 'weekend'])['crime_count'].transform('mean')
panel['month_avg'] = panel.groupby(['precinct', 'crime_type', 'month'])['crime_count'].transform('mean')
panel['count_avg'] = panel.groupby(['precinct', 'crime_type'])['crime_count'].shift(1).fillna(0)
panel['lag7'] = panel.groupby(['precinct', 'crime_type'])['crime_count'].shift(7).fillna(0)
panel['rolling7'] = panel.groupby(['precinct', 'crime_type'])['crime_count'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean()).fillna(0)

panel.to_csv(PANEL_OUT, index=False)
print(panel.shape)
panel.head()

(583968, 26)


,crime_date,precinct,crime_type,crime_count,weekend,holiday,t_mean,rh_mean,wind_mean,has_event,event_match_count,has_road_closure,road_closure_count,is_christian_holiday,is_islamic_holiday,is_jewish_holiday,is_hindu_holiday,month,dow,e_mean,weekday_avg,weekend_avg,month_avg,count_avg,lag7,rolling7
0,2015-01-01,10,ASSAULT,0.0,0.0,0.0,0.6125,37.125,6.625,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,3,2.368922,3.383764,3.020862,2.623656,0.0,0.0,0.000000
1,2015-01-02,10,ASSAULT,2.0,0.0,0.0,3.8125,42.250,6.750,0.0,0.0,1.0,22.0,0.0,0.0,0.0,0.0,1,4,3.384006,3.317343,3.020862,2.623656,0.0,0.0,0.000000
2,2015-01-03,10,ASSAULT,0.0,0.0,0.0,1.7500,74.625,4.250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,5,5.166101,2.963100,3.020862,2.623656,2.0,0.0,1.000000
3,2015-01-04,10,ASSAULT,4.0,1.0,0.0,9.1750,88.875,4.500,0.0,0.0,1.0,32.0,0.0,0.0,0.0,0.0,1,6,10.282045,2.516605,3.242358,2.623656,0.0,0.0,0.666667
4,2015-01-05,10,ASSAULT,0.0,0.0,0.0,2.5000,40.625,8.875,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,2.966370,2.911439,3.020862,2.623656,4.0,0.0,1.500000


In [22]:
centroids = raw.dropna(subset=['latitude', 'longitude']).groupby('precinct')[['latitude', 'longitude']].mean().reindex(precincts)
coords = centroids.to_numpy(dtype=np.float32)
coords = np.nan_to_num(coords, nan=0.0)
adj = build_knn_adjacency(coords, k=K_NEIGHBORS)
adj_norm = normalize_adj(adj)
adj_t = torch.tensor(adj_norm, dtype=torch.float32, device=DEVICE)

print('Precinct count:', len(precincts))
print('Crime types:', crime_types)
print('Adjacency shape:', adj_t.shape)

Precinct count: 77
Crime types: ['ASSAULT', 'CRIMINAL_DAMAGE', 'ROBBERY', 'THEFT']
Adjacency shape: torch.Size([77, 77])


In [23]:
class GraphConv(nn.Module):
    def __init__(self, in_feats, out_feats):
        super().__init__()
        self.lin = nn.Linear(in_feats, out_feats)

    def forward(self, x, adj_norm):
        h = torch.matmul(adj_norm, x)
        return self.lin(h)

class STGCNBlock(nn.Module):
    def __init__(self, in_feats, hidden_feats):
        super().__init__()
        self.gcn = GraphConv(in_feats, hidden_feats)
        self.act = nn.ReLU()

    def forward(self, x, adj_norm):
        return self.act(self.gcn(x, adj_norm))

class ProbSparseAttention(nn.Module):
    def __init__(self, d_model, n_heads=4, dropout=0.1, factor=5):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.factor = factor
        self.scale = self.head_dim ** -0.5
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        b, t, _ = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        sparsity = scores.max(dim=-1).values - scores.mean(dim=-1)
        u = min(t, max(1, int(self.factor * math.log(t + 1))))
        top_idx = torch.topk(sparsity, k=u, dim=-1).indices
        v_mean = v.mean(dim=-2, keepdim=True)
        context = v_mean.repeat(1, 1, t, 1)
        top_scores = scores.gather(-2, top_idx.unsqueeze(-1).repeat(1, 1, 1, t))
        top_attn = torch.softmax(top_scores, dim=-1)
        top_ctx = torch.matmul(top_attn, v)
        context.scatter_(-2, top_idx.unsqueeze(-1).repeat(1, 1, 1, self.head_dim), top_ctx)
        context = context.transpose(1, 2).contiguous().view(b, t, self.d_model)
        return self.out_proj(self.drop(context))

class InformerEncoderLayer(nn.Module):
    def __init__(self, d_model=64, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn = ProbSparseAttention(d_model, n_heads=n_heads, dropout=dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x

class InformerEncoder(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([InformerEncoderLayer(d_model, n_heads, dropout) for _ in range(n_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class HybridInformerSTGCN(nn.Module):
    def __init__(self, node_feat_dim, time_feat_dim, stgcn_hidden=128, d_model=64):
        super().__init__()
        self.stgcn = STGCNBlock(node_feat_dim, stgcn_hidden)
        self.temporal_proj = nn.Linear(time_feat_dim, d_model)
        self.informer = InformerEncoder(d_model=d_model, n_heads=4, n_layers=4)
        self.fuse = nn.Linear(stgcn_hidden + d_model, 1)

    def forward(self, spatial_x, temporal_x, adj_norm):
        b, n, t, f = temporal_x.shape
        spatial_h = self.stgcn(spatial_x, adj_norm)
        temporal_x = temporal_x.reshape(b * n, t, f)
        temporal_h = self.informer(self.temporal_proj(temporal_x))[:, -1, :]
        temporal_h = temporal_h.reshape(b, n, -1)
        fused = torch.cat([spatial_h, temporal_h], dim=-1)
        return self.fuse(fused).squeeze(-1)

In [24]:
def build_series(df_sub, precinct_order, lookback, feature_cols):
    dates = sorted(df_sub['crime_date'].unique())
    n_nodes = len(precinct_order)
    n_features = len(feature_cols)
    series = np.zeros((len(dates), n_nodes, n_features), dtype=np.float32)
    for i, day in enumerate(dates):
        day_df = df_sub[df_sub['crime_date'] == day].set_index('precinct').reindex(precinct_order)
        series[i] = day_df[feature_cols].to_numpy(dtype=np.float32)
    X, y, target_dates = [], [], []
    for i in range(lookback, len(dates)):
        X.append(series[i-lookback:i])
        y.append(series[i, :, 0])
        target_dates.append(dates[i])
    X = np.stack(X) if X else np.zeros((0, lookback, n_nodes, n_features), dtype=np.float32)
    y = np.stack(y) if y else np.zeros((0, n_nodes), dtype=np.float32)
    X = np.transpose(X, (0, 2, 1, 3)) if X.size else np.zeros((0, n_nodes, lookback, n_features), dtype=np.float32)
    return X, y, target_dates

def build_spatial_features(df_sub, precinct_order):
    dates = sorted(df_sub['crime_date'].unique())
    idx = {d: i for i, d in enumerate(dates)}
    series = np.zeros((len(dates), len(precinct_order)), dtype=np.float32)
    for i, day in enumerate(dates):
        day_df = df_sub[df_sub['crime_date'] == day].set_index('precinct').reindex(precinct_order)
        series[i] = day_df['crime_count'].to_numpy(dtype=np.float32)

    def get_at(day, offset):
        j = idx[day] - offset
        if j < 0:
            return np.zeros(len(precinct_order), dtype=np.float32)
        return series[j]

    spatial = []
    for day in dates:
        prox = np.stack([get_at(day, 1), get_at(day, 2), get_at(day, 3)], axis=-1)
        peri = np.stack([get_at(day, 7), get_at(day, 14), get_at(day, 21)], axis=-1)
        trend = get_at(day, 365 * 3)[:, None]
        spatial.append(np.concatenate([prox, peri, trend], axis=-1))
    return np.stack(spatial), dates

def train_model(X, y, spatial_X, adj_norm, epochs=60, batch=32, lr=1e-3, patience=15):
    X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    y_t = torch.tensor(y, dtype=torch.float32, device=DEVICE)
    spatial_t = torch.tensor(spatial_X, dtype=torch.float32, device=DEVICE)
    model = HybridInformerSTGCN(node_feat_dim=spatial_t.shape[-1], time_feat_dim=X_t.shape[-1]).to(DEVICE)
    opt = torch.optim.NAdam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best = float('inf')
    best_state = None
    patience_left = patience

    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(X_t.shape[0])
        epoch_losses = []
        for i in range(0, X_t.shape[0], batch):
            idx = perm[i:i+batch]
            xb = X_t[idx]
            yb = y_t[idx]
            sb = spatial_t[idx]
            opt.zero_grad()
            pred = model(sb, xb, adj_norm)
            loss = rmse_torch(pred, yb)
            loss.backward()
            opt.step()
            epoch_losses.append(loss.item())
        scheduler.step()
        epoch_mean = float(np.mean(epoch_losses)) if epoch_losses else float('inf')
        if epoch_mean < best:
            best = epoch_mean
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def evaluate_model(model, X, y, spatial_X, adj_norm):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        y_t = torch.tensor(y, dtype=torch.float32, device=DEVICE)
        spatial_t = torch.tensor(spatial_X, dtype=torch.float32, device=DEVICE)
        pred = model(spatial_t, X_t, adj_norm)
        mae = torch.mean(torch.abs(pred - y_t)).item()
        rmse = rmse_torch(pred, y_t).item()
    return pred.detach().cpu().numpy(), mae, rmse

In [25]:
feature_candidates = [
    'crime_count', 'weekend', 'holiday', 'weekday_avg', 'weekend_avg', 'month_avg', 'count_avg',
    't_mean', 'rh_mean', 'wind_mean', 'e_mean',
    'has_event', 'event_match_count', 'has_road_closure', 'road_closure_count',
    'is_christian_holiday', 'is_islamic_holiday', 'is_jewish_holiday', 'is_hindu_holiday',
    'lag7', 'rolling7'
]

all_predictions = []
all_metrics = []

for crime_type in crime_types:
    sub = panel[panel['crime_type'] == crime_type].copy()
    feature_cols = [c for c in feature_candidates if c in sub.columns]

    X_all, y_all, target_dates = build_series(sub, precincts, LOOKBACK, feature_cols)
    spatial_all, spatial_dates = build_spatial_features(sub, precincts)
    spatial_idx = {d: i for i, d in enumerate(spatial_dates)}
    spatial_X_all = np.stack([spatial_all[spatial_idx[d]] for d in target_dates])

    train_idx = [i for i, d in enumerate(target_dates) if d <= TRAIN_END]
    test_idx = [i for i, d in enumerate(target_dates) if TEST_START <= d <= TEST_END]

    X_train, y_train, spatial_train = X_all[train_idx], y_all[train_idx], spatial_X_all[train_idx]
    X_test, y_test, spatial_test = X_all[test_idx], y_all[test_idx], spatial_X_all[test_idx]
    test_dates = [target_dates[i] for i in test_idx]

    model = train_model(X_train, y_train, spatial_train, adj_t, epochs=EPOCHS, batch=BATCH_SIZE, lr=LEARNING_RATE, patience=PATIENCE)
    preds, mae, rmse = evaluate_model(model, X_test, y_test, spatial_test, adj_t)

    all_metrics.append({'crime_type': crime_type, 'mae': mae, 'rmse': rmse, 'n_test_days': len(test_dates), 'n_precincts': len(precincts)})

    for i, day in enumerate(test_dates):
        for j, precinct in enumerate(precincts):
            all_predictions.append({
                'crime_type': crime_type,
                'crime_date': day,
                'precinct': precinct,
                'y_true': float(y_test[i, j]),
                'y_pred': float(preds[i, j])
            })

    print(f'{crime_type}: MAE={mae:.4f} RMSE={rmse:.4f}')

pred_df = pd.DataFrame(all_predictions)
summary_df = pd.DataFrame(all_metrics)
pred_df.to_csv(PRED_OUT, index=False)
summary_df.to_csv(SUMMARY_OUT, index=False)

print('Saved predictions to', PRED_OUT)
print('Saved summary to', SUMMARY_OUT)
summary_df

ASSAULT: MAE=2.2009 RMSE=3.0167
CRIMINAL_DAMAGE: MAE=1.5734 RMSE=3.7529
ROBBERY: MAE=0.5894 RMSE=0.7974
THEFT: MAE=1.8190 RMSE=2.3360
Saved predictions to /content/drive/MyDrive/DS340W/final approach/parent_paper_v16c_outputs/parent_paper_predictions_v16c.csv
Saved summary to /content/drive/MyDrive/DS340W/final approach/parent_paper_v16c_outputs/parent_paper_summary_v16c.csv


,crime_type,mae,rmse,n_test_days,n_precincts
0,ASSAULT,2.200917,3.016706,7,77
1,CRIMINAL_DAMAGE,1.573421,3.752851,7,77
2,ROBBERY,0.589358,0.797450,7,77
3,THEFT,1.818996,2.335954,7,77
